# KADMON Optuna — QuickBundles + Partial OT

Les bundles sont compressés en centroïdes QuickBundles pondérés par la taille de leurs clusters, puis comparés avec Partial OT. Cette configuration sert de baseline tractographique pour évaluer les autres méthodes de compression. Le classement utilise `global_distance_mm`; pour chaque bundle, l'objectif minimise le rapport entre la distance intra-identité et la distance moyenne inter-identité.

## 1. Imports et configuration

In [1]:
from pathlib import Path
import os
import sys
from time import perf_counter

import numpy as np
import optuna
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed

WORKING_DIR = Path.cwd().resolve()
KADMON_ROOT = next(
    (path for path in (WORKING_DIR, *WORKING_DIR.parents) if (path / 'kadmon').is_dir() and (path / 'notebooks').is_dir()),
    None,
)
if KADMON_ROOT is None:
    raise RuntimeError(f'Racine KADMON introuvable depuis {WORKING_DIR}.')
NOTEBOOK_DIR = KADMON_ROOT / 'notebooks' / 'optuna'
BUNDLES_DIR = KADMON_ROOT / 'notebooks' / 'bundles'
STUDY_PATH = NOTEBOOK_DIR / 'studies' / 'optuna_reid.sqlite3'
if str(KADMON_ROOT) not in sys.path:
    sys.path.insert(0, str(KADMON_ROOT))

from kadmon.comparison import IneffectiveCompressionError, compare_bundles
from kadmon.io import BundleCollection
from kadmon.protocol import HCP_REID_PROTOCOL
from kadmon.reid import aggregate_reid_metrics, bundle_reid_metrics, set_trial_metrics
from kadmon.selection import select_best_reid_trial
from kadmon.optimization import StudyRunPolicy, create_reid_study, run_until_complete

optuna.logging.set_verbosity(optuna.logging.WARNING)
EXPERIMENT_NAME = 'quickbundles_partial'
COMPRESSION = 'quickbundles'
TRANSPORT = 'partial'
N_TRIALS = 80


Info: some functions in tractosearch.resampling are faster when 'numba' is installed


## 2. Données

Pour chaque bundle, l'acquisition `103818` est comparée aux dix acquisitions `_re`. `103818_re` constitue la comparaison intra-identité et les neuf autres acquisitions les comparaisons inter-identité.

In [2]:
REFERENCE_SUBJECT = HCP_REID_PROTOCOL.reference_subject
INTRA_IDENTITY_SUBJECT = HCP_REID_PROTOCOL.intra_identity_subject
COMPARISON_SUBJECTS = HCP_REID_PROTOCOL.comparison_subjects
SUBJECTS = HCP_REID_PROTOCOL.subjects
N_POINTS = HCP_REID_PROTOCOL.n_points
SEED = 42
BUNDLES_TO_RUN = None  # tuple de noms exacts pour un essai court
N_JOBS_CPU = min(8, os.cpu_count() or 1)
MAX_COST_MATRIX_BYTES = 512 * 2**20  # 512 Mio par comparaison
MAX_REPRESENTATIVES = 5000  # Pruner avant MDF au-delà de cette limite

bundles = BundleCollection(
    BUNDLES_DIR, SUBJECTS, n_points=N_POINTS, selected=BUNDLES_TO_RUN
)
SUBJECT_FILES = bundles.files
bundle_names = bundles.names
bundle_cache = bundles.cache
compression_cache = {}

load_bundle = bundles.load

print(f'Données : {BUNDLES_DIR}')
print('Protocole : 1 comparaison intra-identité et 9 inter-identité par bundle')
print(f'Bundles communs : {len(bundle_names)}; workers CPU : {N_JOBS_CPU}')
display(pd.DataFrame({'bundle': bundle_names}))


Données : /home/colin/Tractographie/KADMON/notebooks/bundles
Protocole : 1 comparaison intra-identité et 9 inter-identité par bundle
Bundles communs : 31; workers CPU : 8


,bundle
0,tractosearch_nn_8_0mm_all_AF_L_m
1,tractosearch_nn_8_0mm_all_AF_R_m
2,tractosearch_nn_8_0mm_all_CC_1_m
3,tractosearch_nn_8_0mm_all_CC_2a_m
4,tractosearch_nn_8_0mm_all_CC_2b_m
5,tractosearch_nn_8_0mm_all_CC_3_m
6,tractosearch_nn_8_0mm_all_CC_4_m
7,tractosearch_nn_8_0mm_all_CC_5_m
8,tractosearch_nn_8_0mm_all_CC_6_m
9,tractosearch_nn_8_0mm_all_CC_7_m


## 3. Espace de recherche

- `threshold`: 6 à 30 mm, par pas de 1 mm;
- `mass`: 0,50 à 1,00, par pas de 0,01.

Le seuil QuickBundles contrôle directement la compression : un seuil plus élevé regroupe davantage de streamlines et produit moins de centroïdes. `mass` fixe la fraction de masse empirique transportée par Partial OT.

In [3]:
def sample_parameters(trial):
    return (
        {'threshold': trial.suggest_float('threshold', 6.0, 30.0, step=1.0)},
        {'mass': trial.suggest_float('mass', 0.50, 1.00, step=0.01)},
    )


## 4. Objectif Optuna

In [4]:
def evaluate_bundle(bundle_name, compression_parameters, transport_parameters):
    source = load_bundle(REFERENCE_SUBJECT, bundle_name)
    pair_metrics = []
    for candidate_subject in COMPARISON_SUBJECTS:
        target = load_bundle(candidate_subject, bundle_name)
        result = compare_bundles(
            source, target, compression=COMPRESSION, transport=TRANSPORT,
            compression_parameters=compression_parameters,
            transport_parameters=transport_parameters,
            compression_cache=compression_cache,
            source_compression_key=(REFERENCE_SUBJECT, bundle_name),
            target_compression_key=(candidate_subject, bundle_name),
            max_cost_matrix_bytes=MAX_COST_MATRIX_BYTES,
            max_representatives=MAX_REPRESENTATIVES,
        )
        metrics = result['metrics']
        pair_metrics.append({
            'global_distance_mm': float(metrics['global_distance_mm']),
            'mean_displacement_mm': float(metrics['mean_mm']),
            'transported_mass': float(metrics['transported_mass']),
            'source_n_representatives': int(metrics['source_n_representatives']),
            'target_n_representatives': int(metrics['target_n_representatives']),
        })

    if len({row['source_n_representatives'] for row in pair_metrics}) != 1:
        raise RuntimeError(f'Compression source non reproductible pour {bundle_name}.')
    return bundle_reid_metrics(
        bundle_name, [row['global_distance_mm'] for row in pair_metrics],
        comparison_subjects=COMPARISON_SUBJECTS,
        intra_identity_subject=INTRA_IDENTITY_SUBJECT,
        mean_displacement_mm=np.mean([r['mean_displacement_mm'] for r in pair_metrics]),
        mean_transported_mass=np.mean([r['transported_mass'] for r in pair_metrics]),
        mean_n_representatives=np.mean([v for r in pair_metrics for v in (r['source_n_representatives'], r['target_n_representatives'])]),
    )

def objective(trial):
    started = perf_counter()
    compression_parameters, transport_parameters = sample_parameters(trial)
    try:
        tasks = (delayed(evaluate_bundle)(name, compression_parameters, transport_parameters) for name in bundle_names)
        results = ([evaluate_bundle(name, compression_parameters, transport_parameters) for name in bundle_names]
                   if N_JOBS_CPU == 1 else
                   Parallel(n_jobs=min(N_JOBS_CPU, len(bundle_names)), backend='threading')(tasks))
    except IneffectiveCompressionError as exc:
        raise optuna.TrialPruned(str(exc)) from exc
    bundle_metrics = pd.DataFrame(results)
    aggregates = aggregate_reid_metrics(
        bundle_metrics, n_comparison_subjects=len(COMPARISON_SUBJECTS),
        elapsed_s=perf_counter() - started,
    )
    set_trial_metrics(trial, aggregates)
    return aggregates['mean_intra_inter_ratio']


## 5. Optimisation

La base SQLite partagée est l'unique sortie automatique de l'étude.

In [5]:
study = create_reid_study(STUDY_PATH, EXPERIMENT_NAME, seed=SEED)
run_until_complete(
    study, objective, StudyRunPolicy(N_TRIALS),
    callbacks=[lambda study, trial: compression_cache.clear()],
)
print(f'Base SQLite : {STUDY_PATH}')


Étude : 80/80 essais COMPLETE; cible restante=0.
Base SQLite : /home/colin/Tractographie/KADMON/notebooks/optuna/studies/optuna_reid.sqlite3


## 6. Analyse des essais observés

In [6]:
trials_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs', 'state'))
complete = trials_df[trials_df['state'] == 'COMPLETE'].dropna(subset=['value'])
display(complete.sort_values('value').head(10))

ratio_best = study.best_trial
reid_best = select_best_reid_trial(study.trials)
display(pd.Series({'experiment': EXPERIMENT_NAME, 'trial': reid_best.number,
                   'selection': 'Top-1, rang, ratio, couverture',
                   'mean_intra_inter_ratio': reid_best.value, **reid_best.params,
                   **reid_best.user_attrs}, name='meilleur essai RE-ID').to_frame())
print(f'Optimum brut du ratio : trial {ratio_best.number}')
print('RE-ID validée :', reid_best.user_attrs.get('reid_valid_bundles', []))
print('RE-ID échouée :', reid_best.user_attrs.get('reid_failed_bundles', []))

optuna.visualization.plot_param_importances(study).show()
optuna.visualization.plot_contour(study, params=['mass', 'threshold']).show()


,number,value,params_mass,params_threshold,user_attrs_elapsed_s,user_attrs_intra_identity_top1_accuracy,user_attrs_mean_displacement_mm,user_attrs_mean_inter_identity_distance_mm,user_attrs_mean_intra_identity_distance_mm,user_attrs_mean_intra_identity_rank,user_attrs_mean_intra_inter_ratio,user_attrs_mean_intra_inter_separation_margin_mm,user_attrs_mean_n_representatives,user_attrs_mean_transported_mass,user_attrs_median_intra_inter_ratio,user_attrs_n_bundles,user_attrs_n_comparisons,user_attrs_reid_failed_bundles,user_attrs_reid_valid_bundles,state
62,62,0.321797,0.90,29.0,13.968863,0.838710,4.691084,5.495047,1.875149,1.387097,0.321797,1.114826,2.285484,0.90,0.268153,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
61,61,0.321797,0.90,29.0,13.981089,0.838710,4.691084,5.495047,1.875149,1.387097,0.321797,1.114826,2.285484,0.90,0.268153,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
60,60,0.321797,0.90,29.0,14.029103,0.838710,4.691084,5.495047,1.875149,1.387097,0.321797,1.114826,2.285484,0.90,0.268153,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
50,50,0.321797,0.90,29.0,14.502189,0.838710,4.691084,5.495047,1.875149,1.387097,0.321797,1.114826,2.285484,0.90,0.268153,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
72,72,0.322464,0.88,29.0,13.962576,0.806452,4.617669,5.376563,1.843294,1.419355,0.322464,1.057421,2.285484,0.88,0.272980,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
85,85,0.322847,0.87,29.0,14.085155,0.806452,4.585263,5.323532,1.828686,1.419355,0.322847,1.027853,2.285484,0.87,0.275062,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
67,67,0.323176,0.91,29.0,14.073035,0.806452,4.732352,5.560567,1.903409,1.451613,0.323176,1.136918,2.285484,0.91,0.272599,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
68,68,0.323176,0.91,29.0,14.024792,0.806452,4.732352,5.560567,1.903409,1.451613,0.323176,1.136918,2.285484,0.91,0.272599,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
79,79,0.323176,0.91,29.0,13.963852,0.806452,4.732352,5.560567,1.903409,1.451613,0.323176,1.136918,2.285484,0.91,0.272599,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
77,77,0.325016,0.92,29.0,13.961480,0.806452,4.775845,5.629331,1.940630,1.451613,0.325016,1.149462,2.285484,0.92,0.268786,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE


,meilleur essai RE-ID
experiment,quickbundles_partial
trial,9
selection,"Top-1, rang, ratio, couverture"
mean_intra_inter_ratio,0.419166
threshold,6.0
mass,0.57
elapsed_s,489.493791
intra_identity_top1_accuracy,0.967742
mean_displacement_mm,3.611114
mean_inter_identity_distance_mm,4.195355


Optimum brut du ratio : trial 50
RE-ID validée : ['tractosearch_nn_8_0mm_all_AF_L_m', 'tractosearch_nn_8_0mm_all_AF_R_m', 'tractosearch_nn_8_0mm_all_CC_1_m', 'tractosearch_nn_8_0mm_all_CC_2a_m', 'tractosearch_nn_8_0mm_all_CC_2b_m', 'tractosearch_nn_8_0mm_all_CC_3_m', 'tractosearch_nn_8_0mm_all_CC_4_m', 'tractosearch_nn_8_0mm_all_CC_5_m', 'tractosearch_nn_8_0mm_all_CC_6_m', 'tractosearch_nn_8_0mm_all_CC_7_m', 'tractosearch_nn_8_0mm_all_CG_L_m', 'tractosearch_nn_8_0mm_all_CG_R_m', 'tractosearch_nn_8_0mm_all_CST_R_m', 'tractosearch_nn_8_0mm_all_ICP_L_m', 'tractosearch_nn_8_0mm_all_ICP_R_m', 'tractosearch_nn_8_0mm_all_IFOF_L_m', 'tractosearch_nn_8_0mm_all_IFOF_R_m', 'tractosearch_nn_8_0mm_all_ILF_L_m', 'tractosearch_nn_8_0mm_all_ILF_R_m', 'tractosearch_nn_8_0mm_all_MCP_m', 'tractosearch_nn_8_0mm_all_OR_L_m', 'tractosearch_nn_8_0mm_all_OR_R_m', 'tractosearch_nn_8_0mm_all_SLF_1_L_m', 'tractosearch_nn_8_0mm_all_SLF_1_R_m', 'tractosearch_nn_8_0mm_all_SLF_2_L_m', 'tractosearch_nn_8_0mm_all_SLF_

In [7]:
tradeoff_df = trials_df.copy()
tradeoff_df = tradeoff_df[tradeoff_df["state"] == "COMPLETE"].dropna(subset=["value"]).rename(columns={
    "number": "trial", "value": "score",
    "params_threshold": "threshold", "params_mass": "mass",
    "user_attrs_mean_n_representatives": "n_representatives",
    "user_attrs_intra_identity_top1_accuracy": "reid_accuracy",
})
best_score = tradeoff_df["score"].min()
rows = []
for limit in (1, 2, 5):
    candidates = tradeoff_df[tradeoff_df["score"] <= best_score * (1 + limit / 100)]
    row = candidates.sort_values(["mass", "score"], ascending=[False, True]).iloc[0]
    rows.append([limit, int(row.trial), row.score, 100 * (row.score / best_score - 1), row.threshold, row.mass, row.n_representatives, row.reid_accuracy])
tradeoff = pd.DataFrame(rows, columns=["seuil (%)", "trial", "score", "écart relatif (%)", "threshold", "mass", "n_representatives", "reid_accuracy"])
display(tradeoff.style.format({"score": "{:.6f}", "écart relatif (%)": "{:.2f}", "threshold": "{:.0f}", "mass": "{:.2f}", "n_representatives": "{:.2f}", "reid_accuracy": "{:.0%}"}))


,seuil (%),trial,score,écart relatif (%),threshold,mass,n_representatives,reid_accuracy
0,1,67,0.323176,0.43,29,0.91,2.29,81%
1,2,65,0.327699,1.83,29,0.93,2.29,81%
2,5,47,0.336107,4.45,29,0.95,2.29,81%


## 7. Interprétation des résultats

L'étude compte **80 essais `COMPLETE`**. Le **trial 50** est l'optimum strict observé (`score=0,321797`, `threshold=29 mm`, `mass=0,90`); les trials 60, 61 et 62 reproduisent exactement ce résultat. Son exactitude RE-ID Top-1 est de **83,87 %** (26 bundles sur 31), son rang intra-identité moyen est de **1,39** et sa marge moyenne de séparation intra–inter est de **1,11 mm**.

Le réglage transporte 90 % de la masse, mais QuickBundles ne produit en moyenne que **2,29 représentants par bundle**. Le seuil optimal est en outre très proche de la borne supérieure testée (`30 mm`). L'étude montre donc qu'une représentation globale extrêmement compacte favorise l'objectif RE-ID utilisé ici; elle ne démontre pas que deux centroïdes suffisent pour décrire des déviations anatomiques locales.

Parmi les essais à moins de 1 % de l'optimum continu, le trial 67 monte à `mass=0,91`, mais son Top-1 baisse à 80,65 %. Selon la règle RE-ID commune, le **trial 9** est le meilleur classement (`threshold=6 mm`, `mass=0,57`, Top-1 96,77 %, rang moyen 1,03 et environ 539 représentants). Pour l'usage anatomique de KADMON, le **trial 75** est toutefois retenu par défaut : `threshold=7 mm`, `mass=0,99`, Top-1 93,55 %, rang moyen 1,10, marge de séparation 2,00 mm et environ 296 représentants. Il transporte presque toute la masse tout en conservant une résolution substantielle.

- Pour la **meilleure RE-ID observée**, utiliser le trial 9 (`threshold=6 mm`, `mass=0,57`).
- Pour les **analyses anatomiques exécutées par défaut dans KADMON**, utiliser le trial 75 (`threshold=7 mm`, `mass=0,99`).
- Le trial 50 reste l'optimum du **ratio continu**, mais ses 2,29 représentants et son Top-1 de 83,87 % le rendent moins pertinent comme défaut.
- Le fait que l'optimum soit proche de la borne supérieure justifie une étude ciblée au-delà de 30 mm uniquement pour confirmer le plateau RE-ID; cela ne rendrait pas la représentation plus anatomiquement détaillée.
- Ces paramètres ne doivent pas être transférés à QuickBundles + Sinkhorn, qui possède son propre compromis entre compression et régularisation.